# Phase 1 of 3: Data Generation (v3)

**Run order:** 1 of 4 notebooks (`phase_1` → `phase_2` → `phase_3` → `phase_4_eval`).
**Needs:** network access to Hugging Face on the first run (later runs read `data/source-raw/`).
**Produces:** `data/source-clean-v3/{train,val,test,adversarial_test,lora_holdout}.parquet` and `manifest.json`. v1 and v2 stay frozen in their own folders.
**Expect:** the first run downloads about 2 GB (Nemotron is the bulk); MinHash over roughly 90k rows takes a few minutes on CPU; the last cell must print `leakage checks passed` and a counts line per output file.

Phase 1 of `docs/PLAN.md`: build a clean, deduplicated, leakage-safe binary dataset
(injection vs. safe, with a reason label on positives) from Hugging Face sources only, plus a
locked adversarial evaluation set and the LoRA's held-out comparison set.

**What v3 changes, and why (README "Progress and findings", v2 run of 2026-10-09):**

- **BIPIA documents are keyed by whitespace-normalized base text.** v2 hashed the raw base text;
  removing an attack from the middle of a document leaves different whitespace than removing it
  from the end, so one document became up to three ids and 18 base documents were still attacked
  in both train and test. BIPIA has only 182 distinct base documents (47 emails, 84 tables, 51 code).
- **Injected BIPIA rows are capped at three per base document** (v2: 48 per attack string, which
  gave 5,792 injected rows against 146 clean ones in train, a 40:1 prior that taught the model
  "BIPIA-style document" rather than "attack inside document"). BIPIA is now a small, honest
  indirect-injection slice; its clean-document false-positive rate is reported with its row count.

**What v2 changed (kept in v3):**

1. **BIPIA is partitioned jointly by base document and attack string.** v1 kept each attack
   string in one split but let a base document appear attacked in train and clean in test; the
   model learned the documents and flagged 61% of clean BIPIA documents. v2 assigns documents and
   attack strings to splits independently and keeps only injected rows whose document and attack
   string landed in the same split, so a test row has an unseen document *and* an unseen attack.
2. **Structured benign documents from outside BIPIA**: legitimate emails from
   `puyang2025/seven-phishing-email-datasets` (label 0), the corpus the sibling email classifier
   uses, so "looks like an email" cannot stand in for "contains an attack".
3. **Tighter template grouping**: leading-template key shortened from 12 to 8 words and the
   MinHash near-duplicate threshold lowered from 0.90 to 0.80, so paraphrase families of one
   jailbreak skeleton share a split. v1 left a 42% six-word twin rate between train and test.
4. New dataset version, new fingerprint; nothing in earlier versions is overwritten.

**Sources (verified on Hugging Face 2026-10-09):**

| Source | Rows used | Label | Reason | Shape |
|---|---|---|---|---|
| `nvidia/Nemotron-Safety-Guard-Dataset-v3`, train split, English, tag `jailbreaking` | 10,000 | INJECTION | Jailbreak Technique | chat |
| `allenai/wildguardmix` `wildguardtrain`, adversarial and harmful | 20,567 | INJECTION | LoRA subcategory map | chat |
| `allenai/wildguardmix` `wildguardtrain`, unharmful | 40,543 | SAFE | — | chat |
| `deepset/prompt-injections` (en/de) | 662 | by label | Override Attempt | chat |
| `jackhhao/jailbreak-classification` | 1,306 | by type | Jailbreak Technique | chat |
| `geodesic-research/bipia` (indirect injection in emails, tables, code) | up to 3 per base document, document and attack string in the same split | INJECTION | Embedded Instruction | document |
| `geodesic-research/bipia`, attack string removed | one per base document | SAFE | — | document |
| `puyang2025/seven-phishing-email-datasets`, label 0 (legitimate email) | up to 3,000 | SAFE | — | document |
| `wikimedia/wikipedia` `20231101.en`, streamed paragraphs | up to 6,000 | SAFE | — | document |

Rejected after verification: `xTRam1/safe-guard-prompt-injection` (no license on the card).
Deferred: `nvidia/Nemotron-RL-Agentic-Indirect-Prompt-Injection-v1` (raw agent traces, no documented schema).

**Held out, never trained on:** `wildguardtest` (labelled exactly as the LoRA eval notebook labels
it) and the Nemotron test-split English `jailbreaking` rows. Pool rows that match a held-out row
exactly or as a near-duplicate are dropped. The held-out rows are saved as `lora_holdout.parquet`
for `notebooks/eval/`.

**Labelling carried over from the LoRA, by design:** WildGuardMix rows that are adversarial but
unharmful are SAFE, as in `safety/notebooks/prompt_injection_lora_qwen38_27b.ipynb`, so the two
detectors are graded on the same contract. They are kept as their own source slice
(`wildguard_benign_adversarial`) so their false-positive rate is visible. Nemotron `jailbreaking`
rows are INJECTION regardless of their `prompt_label`; that label is kept in `attack_type`.

**Reason labels are weak.** Every reason is map-derived from a source or subcategory, not human
annotated. Four of the nine reasons have training rows after this build (Jailbreak Technique,
Privilege Escalation, Meta-Command Injection, Override Attempt, Embedded Instruction is the fifth);
the rest are empty until Phase 4. The manifest records the per-split reason distribution.

**Safety note:** the data contains live attack strings. Never paste them into a real model
session outside these notebooks.

**Outputs:** `data/source-clean-v1/{train,val,test,adversarial_test,lora_holdout}.parquet`, `manifest.json`.


## 1. Configuration


In [ ]:
import hashlib, json, os
from pathlib import Path

SEED = 42
DATASET_VERSION = "v3"        # v1 and v2 outputs are frozen in their own folders, never overwritten

# Resolve the project root so this notebook runs unchanged on the DGX Spark (container or
# host), a Mac, or any other checkout: env var first, then the Spark paths, then walk up
# from the working directory until a folder containing docs/PLAN.md is found.
def resolve_project_dir():
    candidates = [
        Path(p) for p in (
            os.environ.get("PI_NN_DIR"),
            "/workspace/training/prompt-injection-nn",
            "/home/spark/projects/training/prompt-injection-nn",
        ) if p
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    for parent in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (parent / "docs" / "PLAN.md").exists() and (parent / "notebooks").is_dir():
            return parent
    raise FileNotFoundError(f"project root not found; tried {candidates} and parents of {Path.cwd()}")


PROJECT_DIR = resolve_project_dir()
RAW_DIR = PROJECT_DIR / "data" / "source-raw"
CLEAN_DIR = PROJECT_DIR / "data" / f"source-clean-{DATASET_VERSION}"
RAW_DIR.mkdir(parents=True, exist_ok=True)
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

# Hugging Face sources. Schemas, licenses and splits verified 2026-10-09; see the table above.
HF_NEMOTRON = "nvidia/Nemotron-Safety-Guard-Dataset-v3"
HF_WILDGUARD = "allenai/wildguardmix"           # configs: wildguardtrain (train), wildguardtest (test); gated, auto-approved
HF_DEEPSET = "deepset/prompt-injections"       # text, label (1 = injection); splits train/test
HF_JACKHHAO = "jackhhao/jailbreak-classification"  # prompt, type (jailbreak | benign); splits train/test
HF_BIPIA = "geodesic-research/bipia"           # context already contains attack_str at `position`
HF_WIKIPEDIA = "wikimedia/wikipedia"
HF_EMAIL = "puyang2025/seven-phishing-email-datasets"  # text, subject, label (0 = legitimate), dataset_name; shared with email-classifier-nn
WIKIPEDIA_CONFIG = "20231101.en"
HF_DATASETS = (HF_NEMOTRON, f"{HF_WILDGUARD}/wildguardtrain", f"{HF_WILDGUARD}/wildguardtest",
               HF_DEEPSET, HF_JACKHHAO, HF_BIPIA, f"{HF_WIKIPEDIA}/{WIKIPEDIA_CONFIG}", HF_EMAIL)
EXCLUDED_SOURCES = {
    "xTRam1/safe-guard-prompt-injection": "no license on the dataset card",
    "nvidia/Nemotron-RL-Agentic-Indirect-Prompt-Injection-v1":
        "raw agent tool-call traces without a documented schema; deferred to a later data version",
}

# Caps. 0 means every eligible row. Applied with a seeded sample after filtering, before dedupe.
NEMOTRON_MAX = 0
WILDGUARD_INJECTION_MAX = 0
WILDGUARD_SAFE_MAX = 0
BIPIA_ROWS_PER_DOC = 3                # 182 base documents x up to 3 attacks each (v2: 48 per attack string, 40:1 injected:clean)
WIKIPEDIA_MAX_ARTICLES = 1500         # streamed in dataset order, deterministic
WIKIPEDIA_MAX_PARAGRAPHS = 6000
WIKIPEDIA_MIN_PARAGRAPH_CHARS = 200
EMAIL_HAM_MAX = 3000                  # legitimate emails as structured document-shaped SAFE rows
EMAIL_MIN_BODY_CHARS = 200
BIPIA_SPLIT_SEED = SEED + 7           # independent 80/10/10 partitions of BIPIA documents and attack strings

MIN_TEXT_CHARS = 8                    # "Ignore the above" is a valid 16-char injection; keep the gate low
NEAR_DUP_THRESHOLD = 0.80             # MinHash LSH Jaccard threshold (v1: 0.90)
MINHASH_PERMS = 128
SHINGLE_SIZE = 5                      # word shingles
TEMPLATE_KEY_WORDS = 8                # chat-shaped positives sharing this many leading words share a split (v1: 12)
SPLITS = {"train": 0.80, "val": 0.10, "test": 0.10}
AUG_FRACTION = 0.25                   # fraction of train positives that get an obfuscated copy

# Output contract shared with the LoRA and the Open WebUI filter: "SAFE" or "INJECTION: <reason>".
REASONS = [
    "Override Attempt",
    "Fake System Injection",
    "Meta-Command Injection",
    "Jailbreak Technique",
    "Delimiter Confusion",
    "Privilege Escalation",
    "Embedded Instruction",
    "Role-Play Bypass",
    "Chain-of-Instructions",
]

# Copied from safety/notebooks/prompt_injection_lora_qwen38_27b.ipynb. WildGuardMix subcategories
# not listed here map to Jailbreak Technique, as in the LoRA.
SUBCATEGORY_REASON_MAP = {
    "cyberattack": "Meta-Command Injection",
    "fraud_assisting_illegal_activities": "Jailbreak Technique",
    "sensitive_information_organization_government": "Privilege Escalation",
    "private_information_individual": "Privilege Escalation",
    "copyright_violations": "Jailbreak Technique",
    "defamation_encouraging_unethical_or_unsafe_actions": "Jailbreak Technique",
    "others": "Jailbreak Technique",
}
DEFAULT_REASON = "Jailbreak Technique"

CONFIG_FINGERPRINT = hashlib.sha256(json.dumps({
    "version": DATASET_VERSION, "datasets": HF_DATASETS, "excluded": EXCLUDED_SOURCES,
    "caps": {"nemotron": NEMOTRON_MAX, "wg_inj": WILDGUARD_INJECTION_MAX, "wg_safe": WILDGUARD_SAFE_MAX,
             "bipia_per_doc": BIPIA_ROWS_PER_DOC, "wiki_articles": WIKIPEDIA_MAX_ARTICLES,
             "wiki_paragraphs": WIKIPEDIA_MAX_PARAGRAPHS, "wiki_min_chars": WIKIPEDIA_MIN_PARAGRAPH_CHARS,
             "email_ham": EMAIL_HAM_MAX, "email_min_chars": EMAIL_MIN_BODY_CHARS, "bipia_split_seed": BIPIA_SPLIT_SEED},
    "min_chars": MIN_TEXT_CHARS, "near_dup": NEAR_DUP_THRESHOLD, "perms": MINHASH_PERMS,
    "shingle": SHINGLE_SIZE, "template_words": TEMPLATE_KEY_WORDS, "splits": SPLITS,
    "aug": AUG_FRACTION, "seed": SEED, "reasons": REASONS, "subcategory_map": SUBCATEGORY_REASON_MAP,
    "split_alloc": "balanced-rows-label-source-reason-shape; bipia joint doc+template partition, normalized doc key",
}, sort_keys=True).encode()).hexdigest()
print("dataset version:", DATASET_VERSION)
print("output dir:", CLEAN_DIR)
print("config fingerprint:", CONFIG_FINGERPRINT[:16])


## 2. Environment


In [ ]:
%pip install -q datasets pandas pyarrow datasketch

import gc, re, unicodedata
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from datasets import load_dataset
from datasketch import MinHash, MinHashLSH

WS_RE = re.compile(r"\s+")
WORD_RE = re.compile(r"[a-z0-9]+")


def normalize_for_fp(text):
    return WS_RE.sub(" ", (text or "").lower()).strip()


def fingerprint(text):
    return hashlib.sha256(normalize_for_fp(text).encode("utf-8", "replace")).hexdigest()


def template_key(text, words=TEMPLATE_KEY_WORDS):
    # Leading words of the normalized text. Jailbreak corpora are generated from a few dozen
    # skeletons ("From now on you are going to act as ..."); rows sharing a skeleton share a split.
    tokens = WORD_RE.findall(normalize_for_fp(text))
    return " ".join(tokens[:words]) if len(tokens) >= words else None


def short_hash(text):
    return hashlib.sha256(text.encode("utf-8", "replace")).hexdigest()[:16]


## 3. Download Hugging Face Sources

Every source is loaded with `datasets.load_dataset` and cached as parquet in `data/source-raw/`;
re-runs read the local cache. Only the rows this build uses are cached (for Nemotron, the English
`jailbreaking` rows of the train and test splits, not the 450k-row corpus). Wikipedia is streamed
in dataset order and cut off after `WIKIPEDIA_MAX_ARTICLES`, which is deterministic.


In [ ]:
def load_hf_cache(name, cache_path, build):
    if cache_path.exists():
        frame = pd.read_parquet(cache_path)
        print(f"{name} (cached): {len(frame):,} rows")
        return frame
    frame = build()
    tmp = Path(str(cache_path) + ".tmp")
    frame.to_parquet(tmp, index=False)
    os.replace(tmp, cache_path)
    gc.collect()
    print(f"{name} (downloaded): {len(frame):,} rows")
    return frame


def nemotron_jailbreaking(split):
    def build():
        dataset = load_dataset(HF_NEMOTRON, split=split)
        kept = dataset.filter(
            lambda x: x.get("tag") == "jailbreaking"
            and x.get("language") in (None, "en")
            and x.get("prompt") not in (None, "REDACTED")
        )
        return kept.to_pandas()[["id", "prompt", "prompt_label"]]
    return load_hf_cache(f"{HF_NEMOTRON}:{split}:jailbreaking:en", RAW_DIR / f"nemotron_{split}_jailbreaking_en.parquet", build)


def wildguard(config, split):
    cols = ["prompt", "adversarial", "prompt_harm_label", "subcategory"]
    return load_hf_cache(f"{HF_WILDGUARD}/{config}", RAW_DIR / f"wildguard_{config}.parquet",
                         lambda: load_dataset(HF_WILDGUARD, config, split=split).to_pandas()[cols])


def with_splits(dataset_id, cache_name):
    def build():
        dd = load_dataset(dataset_id)
        parts = []
        for split_name, part in dd.items():
            frame = part.to_pandas()
            frame["hf_split"] = split_name
            parts.append(frame)
        return pd.concat(parts, ignore_index=True)
    return load_hf_cache(dataset_id, RAW_DIR / f"{cache_name}.parquet", build)


def bipia():
    cols = ["id", "task_name", "context", "attack_name", "attack_str", "attack_category", "position"]
    return load_hf_cache(HF_BIPIA, RAW_DIR / "bipia.parquet",
                         lambda: load_dataset(HF_BIPIA, split="train").to_pandas()[cols])


def wikipedia_articles():
    def build():
        stream = load_dataset(HF_WIKIPEDIA, WIKIPEDIA_CONFIG, split="train", streaming=True)
        rows = []
        for row in stream:
            rows.append({"id": row["id"], "title": row["title"], "text": row["text"]})
            if len(rows) >= WIKIPEDIA_MAX_ARTICLES:
                break
        return pd.DataFrame(rows)
    return load_hf_cache(f"{HF_WIKIPEDIA}/{WIKIPEDIA_CONFIG}", RAW_DIR / f"wikipedia_{WIKIPEDIA_CONFIG}_first{WIKIPEDIA_MAX_ARTICLES}.parquet", build)


def email_ham():
    def build():
        dataset = load_dataset(HF_EMAIL, split="train")
        kept = dataset.filter(lambda x: x.get("label") == 0 and isinstance(x.get("text"), str) and len(x["text"].strip()) >= EMAIL_MIN_BODY_CHARS)
        return kept.to_pandas()[["text", "subject", "dataset_name"]]
    return load_hf_cache(f"{HF_EMAIL}:label0", RAW_DIR / "email_ham.parquet", build)


raw = {
    "email_ham": email_ham(),
    "nemotron_train": nemotron_jailbreaking("train"),
    "nemotron_test": nemotron_jailbreaking("test"),
    "wildguard_train": wildguard("wildguardtrain", "train"),
    "wildguard_test": wildguard("wildguardtest", "test"),
    "deepset": with_splits(HF_DEEPSET, "deepset_prompt_injections"),
    "jackhhao": with_splits(HF_JACKHHAO, "jackhhao_jailbreak_classification"),
    "bipia": bipia(),
    "wikipedia": wikipedia_articles(),
}
for name, frame in raw.items():
    print(f"{name:16s} {len(frame):>7,} rows  columns={frame.columns.tolist()}")


## 4. Normalize

Every source is mapped to `text, label, reason, source, shape, synthetic, attack_type, hf_dataset,
source_id, group_key`. `shape` is source-derived (chat prompt vs document chunk), recorded so the
split allocator can balance it and the evaluation can slice on it. `group_key` is set only where a
source has an explicit template or document identity (BIPIA, keyed by normalized text); everything else is grouped by
near-duplicate cluster and leading-words template key in section 7.


In [ ]:
COMMON_COLS = ["text", "label", "reason", "source", "shape", "synthetic", "attack_type", "hf_dataset", "source_id", "group_key", "forced_split", "doc_key"]


def values(x):
    # Series are taken by position, never aligned by index: the source frames carry arbitrary indices.
    return x.to_numpy() if isinstance(x, pd.Series) else x


def rows(frame, text_col, label, reason, source, shape, hf_dataset, source_id, attack_type=None, group_key=None, forced_split=None, doc_key=None):
    out = pd.DataFrame({
        "text": frame[text_col].fillna("").astype(str).to_numpy(),
        "label": int(label),
        "reason": reason if label else "",
        "source": source,
        "shape": shape,
        "synthetic": False,
        "attack_type": values(attack_type) if attack_type is not None else "",
        "hf_dataset": hf_dataset,
        "source_id": values(source_id),
        "group_key": values(group_key) if group_key is not None else None,
        "forced_split": values(forced_split) if forced_split is not None else None,
        "doc_key": values(doc_key) if doc_key is not None else None,
    })
    return out[COMMON_COLS]


def seeded_cap(frame, cap):
    return frame if not cap or len(frame) <= cap else frame.sample(n=cap, random_state=SEED)


parts = []

# Nemotron: every jailbreaking row is INJECTION (LoRA labelling); prompt_label kept for the slice.
nm = seeded_cap(raw["nemotron_train"], NEMOTRON_MAX)
parts.append(rows(nm, "prompt", 1, "Jailbreak Technique", "nemotron_jailbreaking", "chat", HF_NEMOTRON,
                  "nemotron:" + nm["id"].astype(str), attack_type="jailbreaking/" + nm["prompt_label"].fillna("").astype(str)))

# WildGuardMix train. Adversarial + harmful -> INJECTION with the LoRA's subcategory map.
# Unharmful -> SAFE, split into two source slices by the adversarial flag.
wg = raw["wildguard_train"].copy()
wg["source_id"] = "wildguardtrain:" + wg.index.astype(str)
wg_inj = seeded_cap(wg[(wg["adversarial"] == True) & (wg["prompt_harm_label"] == "harmful")], WILDGUARD_INJECTION_MAX)
wg_inj_reason = wg_inj["subcategory"].fillna("others").map(lambda s: SUBCATEGORY_REASON_MAP.get(s, DEFAULT_REASON))
frame = rows(wg_inj, "prompt", 1, "", "wildguard_adversarial", "chat", f"{HF_WILDGUARD}/wildguardtrain",
             wg_inj["source_id"], attack_type=wg_inj["subcategory"].fillna("others"))
frame["reason"] = wg_inj_reason.to_numpy()
parts.append(frame)
wg_safe = seeded_cap(wg[wg["prompt_harm_label"] == "unharmful"], WILDGUARD_SAFE_MAX)
for adversarial, source in ((False, "wildguard_benign"), (True, "wildguard_benign_adversarial")):
    sub = wg_safe[wg_safe["adversarial"] == adversarial]
    parts.append(rows(sub, "prompt", 0, "", source, "chat", f"{HF_WILDGUARD}/wildguardtrain", sub["source_id"],
                      attack_type=sub["subcategory"].fillna("benign")))

# deepset: short direct injections (en/de). Positives are overrides of the preceding instructions.
dp = raw["deepset"].copy()
dp["source_id"] = "deepset:" + dp["hf_split"] + ":" + dp.index.astype(str)
parts.append(rows(dp[dp["label"] == 1], "text", 1, "Override Attempt", "deepset_injection", "chat", HF_DEEPSET, dp.loc[dp["label"] == 1, "source_id"]))
parts.append(rows(dp[dp["label"] == 0], "text", 0, "", "deepset_benign", "chat", HF_DEEPSET, dp.loc[dp["label"] == 0, "source_id"]))

# jackhhao: DAN-style jailbreak prompts vs benign role-play prompts.
jh = raw["jackhhao"].copy()
jh["source_id"] = "jackhhao:" + jh["hf_split"] + ":" + jh.index.astype(str)
parts.append(rows(jh[jh["type"] == "jailbreak"], "prompt", 1, "Jailbreak Technique", "jackhhao_jailbreak", "chat", HF_JACKHHAO, jh.loc[jh["type"] == "jailbreak", "source_id"]))
parts.append(rows(jh[jh["type"] == "benign"], "prompt", 0, "", "jackhhao_benign", "chat", HF_JACKHHAO, jh.loc[jh["type"] == "benign", "source_id"]))

# BIPIA: the indirect case. `context` already carries `attack_str` at `position` (asserted here).
# Injected rows are grouped by attack template so a template graded in test was never trained on.
# Clean documents are recovered by removing the attack string once; one SAFE row per base document.
bp = raw["bipia"].copy()
bp["context"] = bp["context"].fillna("").astype(str)
bp["attack_str"] = bp["attack_str"].fillna("").astype(str)
assert (bp["attack_str"].str.len() > 0).all(), "BIPIA row without an attack string"
inside = [a in c for a, c in zip(bp["attack_str"], bp["context"])]
assert all(inside), f"{len(inside) - sum(inside)} BIPIA rows do not contain their attack string in context"
bp["base_doc"] = [c.replace(a, "", 1).strip() for a, c in zip(bp["attack_str"], bp["context"])]
bp["template_id"] = bp["attack_str"].map(short_hash)
bp["doc_id"] = bp["base_doc"].map(lambda t: short_hash(normalize_for_fp(t)))  # v3: normalized, so start/middle/end variants share one id


def partition(keys, seed):
    # Independent, seeded 80/10/10 assignment of a set of ids to splits.
    keys = sorted(set(keys))
    order = np.random.default_rng(seed).permutation(len(keys))
    bounds = np.cumsum([SPLITS[s] for s in SPLITS]) * len(keys)
    assignment = {}
    for rank, index in enumerate(order):
        split = next(s for s, bound in zip(SPLITS, bounds) if rank < bound - 1e-9) if rank < bounds[-1] else list(SPLITS)[-1]
        assignment[keys[index]] = split
    return assignment


doc_split = partition(bp["doc_id"], BIPIA_SPLIT_SEED)
template_split = partition(bp["template_id"], BIPIA_SPLIT_SEED + 1)
bp["doc_split"] = bp["doc_id"].map(doc_split)
bp["template_split"] = bp["template_id"].map(template_split)
# v2: an injected row is usable only when its document and its attack string landed in the same
# split. Test rows therefore carry an unseen document and an unseen attack string.
bp_same = bp[bp["doc_split"] == bp["template_split"]]
bp_inj = bp_same.sample(frac=1.0, random_state=SEED).groupby("doc_id").head(BIPIA_ROWS_PER_DOC)
parts.append(rows(bp_inj, "context", 1, "Embedded Instruction", "bipia_injected", "document", HF_BIPIA, "bipia:" + bp_inj["id"].astype(str),
                  attack_type=bp_inj["task_name"] + "/" + bp_inj["attack_category"] + "/" + bp_inj["attack_name"] + "/" + bp_inj["position"],
                  group_key="bipia_template:" + bp_inj["template_id"], forced_split=bp_inj["template_split"], doc_key="bipia_doc:" + bp_inj["doc_id"]))
bp_clean = bp.drop_duplicates("doc_id")
parts.append(rows(bp_clean, "base_doc", 0, "", "bipia_clean", "document", HF_BIPIA, "bipia_clean:" + bp_clean["doc_id"],
                  attack_type=bp_clean["task_name"], group_key="bipia_doc:" + bp_clean["doc_id"], forced_split=bp_clean["doc_split"], doc_key="bipia_doc:" + bp_clean["doc_id"]))
print(f"BIPIA: {bp['template_id'].nunique()} attack templates, {bp['doc_id'].nunique()} base documents; "
      f"{len(bp_same):,} rows with document and template in the same split, {len(bp_inj):,} injected rows sampled, {len(bp_clean):,} clean documents")
print("BIPIA injected rows per split:", bp_inj["template_split"].value_counts().to_dict(), "| clean docs per split:", bp_clean["doc_split"].value_counts().to_dict())

# Legitimate emails: structured, document-shaped SAFE rows that are not BIPIA, so document shape
# alone, and BIPIA's house style in particular, cannot stand in for "contains an attack".
em = raw["email_ham"].copy()
em["text"] = em["text"].fillna("").astype(str)
em["subject"] = em["subject"].fillna("").astype(str)
em["doc"] = np.where(em["subject"].str.strip().str.len() > 0, "Subject: " + em["subject"].str.strip() + "\n\n" + em["text"], em["text"])
em = seeded_cap(em, EMAIL_HAM_MAX)
em["source_id"] = "email_ham:" + em.index.astype(str)
parts.append(rows(em, "doc", 0, "", "email_ham", "document", HF_EMAIL, em["source_id"], attack_type=em["dataset_name"].fillna("").astype(str)))

# Wikipedia: prose-shaped SAFE paragraphs so document shape alone is not an injection signal.
wk_rows = []
for _, article in raw["wikipedia"].iterrows():
    for k, paragraph in enumerate(str(article["text"]).split("\n\n")):
        paragraph = paragraph.strip()
        if len(paragraph) >= WIKIPEDIA_MIN_PARAGRAPH_CHARS and not paragraph.startswith("=="):
            wk_rows.append({"text": paragraph, "source_id": f"wikipedia:{article['id']}#{k}", "doc": str(article["id"])})
wk = pd.DataFrame(wk_rows)
wk = wk.iloc[np.random.default_rng(SEED).permutation(len(wk))[:WIKIPEDIA_MAX_PARAGRAPHS]]
parts.append(rows(wk, "text", 0, "", "wikipedia_paragraph", "document", f"{HF_WIKIPEDIA}/{WIKIPEDIA_CONFIG}", wk["source_id"],
                  group_key="wikipedia_article:" + wk["doc"]))

df = pd.concat(parts, ignore_index=True)
df["text"] = df["text"].fillna("").astype(str)
df["label"] = pd.to_numeric(df["label"], errors="coerce")
df["holdout"] = False

# Held-out comparison set, labelled exactly as safety/notebooks/prompt_injection_lora_qwen38_27b_eval.ipynb labels it.
wt = raw["wildguard_test"].copy()
wt["source_id"] = "wildguardtest:" + wt.index.astype(str)
wt_safe = wt[wt["prompt_harm_label"] == "unharmful"]
wt_inj = wt[(wt["adversarial"] == True) & (wt["prompt_harm_label"] == "harmful")]
holdout_parts = [
    rows(wt_safe, "prompt", 0, "", "wildguard_test", "chat", f"{HF_WILDGUARD}/wildguardtest", wt_safe["source_id"], attack_type=wt_safe["subcategory"].fillna("benign")),
    rows(wt_inj, "prompt", 1, "", "wildguard_test", "chat", f"{HF_WILDGUARD}/wildguardtest", wt_inj["source_id"], attack_type=wt_inj["subcategory"].fillna("others")),
]
holdout_parts[1]["reason"] = wt_inj["subcategory"].fillna("others").map(lambda s: SUBCATEGORY_REASON_MAP.get(s, DEFAULT_REASON)).to_numpy()
nt = raw["nemotron_test"]
holdout_parts.append(rows(nt, "prompt", 1, "Jailbreak Technique", "nemotron_test_jailbreaking", "chat", HF_NEMOTRON,
                          "nemotron_test:" + nt["id"].astype(str), attack_type="jailbreaking/" + nt["prompt_label"].fillna("").astype(str)))
holdout = pd.concat(holdout_parts, ignore_index=True)
holdout["text"] = holdout["text"].fillna("").astype(str)
holdout["holdout"] = True

print(df.groupby(["source", "label"]).size().unstack(fill_value=0))
print()
print("shape x label:")
print(df.groupby(["shape", "label"]).size().unstack(fill_value=0))
print()
print("held-out rows:", holdout.groupby(["source", "label"]).size().to_dict())
print(f"pool rows before quality gates: {len(df):,}")


## 5. Quality Gates

Drop rows with an invalid label, a reason outside the contract, or near-empty text. Applied to the pool and to the held-out set.


In [ ]:
def quality_gate(frame, name):
    n0 = len(frame)
    frame = frame[frame["label"].isin([0, 1])].copy()
    frame["label"] = frame["label"].astype(int)
    valid_reason = ((frame["label"] == 0) & (frame["reason"] == "")) | ((frame["label"] == 1) & frame["reason"].isin(REASONS))
    bad = int((~valid_reason).sum())
    assert bad == 0, f"{name}: {bad} rows violate the reason contract"
    frame = frame[frame["text"].str.strip().str.len() >= MIN_TEXT_CHARS].copy()
    frame = frame.reset_index(drop=True)
    print(f"{name} quality gates: {n0:,} -> {len(frame):,} rows ({n0 - len(frame):,} dropped)")
    return frame


df = quality_gate(df, "pool")
holdout = quality_gate(holdout, "holdout")


## 6. Deduplication and Held-Out Isolation

1. **Exact**: SHA-256 of whitespace-normalized, lowercased text. Pool rows that match a held-out
   row are dropped.
2. **Near-dup**: MinHash LSH over word 5-gram shingles, clustered with union-find, built over the
   pool **and** the held-out rows together. Any pool row that lands in a cluster with a held-out row
   is dropped, so the LoRA comparison set shares no paraphrase with training data. Clusters are
   later used as split-grouping keys.

BIPIA rows are excluded from the LSH: by construction every injected row is a near-duplicate of
its base document and of every other attack on that document, which would merge the corpus into one
cluster. They carry explicit template and document keys instead (section 4).


In [ ]:
df["fp"] = df["text"].map(fingerprint)
holdout["fp"] = holdout["text"].map(fingerprint)
n0 = len(df)
df = df.drop_duplicates(subset="fp", keep="first").reset_index(drop=True)
print(f"exact dedup (pool): {n0:,} -> {len(df):,}")
n0 = len(holdout)
holdout = holdout.drop_duplicates(subset="fp", keep="first").reset_index(drop=True)
print(f"exact dedup (holdout): {n0:,} -> {len(holdout):,}")
n0 = len(df)
df = df[~df["fp"].isin(set(holdout["fp"]))].reset_index(drop=True)
print(f"pool rows identical to a held-out row dropped: {n0 - len(df):,}")


def word_shingles(text, k=SHINGLE_SIZE):
    words = normalize_for_fp(text).split()
    if len(words) < k:
        return {" ".join(words)} if words else set()
    return {" ".join(words[i:i + k]) for i in range(len(words) - k + 1)}


combined = pd.concat([df[["text", "source", "holdout"]], holdout[["text", "source", "holdout"]]], ignore_index=True)
n_pool = len(df)
parent = list(range(len(combined)))


def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x


def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[rb] = ra


lsh = MinHashLSH(threshold=NEAR_DUP_THRESHOLD, num_perm=MINHASH_PERMS)
signatures = {}
skip_lsh = combined["source"].str.startswith("bipia_").to_numpy()
for i, text in enumerate(combined["text"]):
    if skip_lsh[i]:
        continue
    mh = MinHash(num_perm=MINHASH_PERMS)
    for sh in word_shingles(text):
        mh.update(sh.encode("utf-8", "replace"))
    signatures[i] = mh
    lsh.insert(str(i), mh)
    if (i + 1) % 20000 == 0:
        print(f"  minhash {i + 1:,}/{len(combined):,}")

for i, mh in signatures.items():
    for key in lsh.query(mh):
        j = int(key)
        if j != i:
            union(i, j)

combined["dup_cluster"] = [find(i) for i in range(len(combined))]
holdout_clusters = set(combined.loc[combined["holdout"], "dup_cluster"])
pool_clusters = combined.loc[:n_pool - 1, "dup_cluster"].to_numpy()
near_holdout = np.isin(pool_clusters, list(holdout_clusters))
df["dup_cluster"] = pool_clusters
n0 = len(df)
df = df[~near_holdout].reset_index(drop=True)
print(f"pool rows that are near-duplicates of a held-out row dropped: {n0 - len(df):,}")
print(f"near-dup clusters (pool): {df['dup_cluster'].nunique():,} for {len(df):,} rows")
del signatures, lsh, combined, parent
gc.collect()


## 7. Leakage-Safe Splits

Grouping key = union of (a) near-dup cluster, (b) explicit `group_key` where a source has one
(BIPIA template or document, Wikipedia article), and (c) for chat-shaped positives, the leading
`TEMPLATE_KEY_WORDS` words, so prompts generated from one jailbreak skeleton never straddle splits.
The ten largest groups are printed; a single giant group would make the split degenerate and is
something to look at, not hide.

Groups are assigned greedily (largest first, seeded shuffle among ties) by a marginal-cost rule that
balances rows, labels, every source, every reason, and both shapes against the 80/10/10 targets.
No group spans splits by construction.


In [ ]:
parent = list(range(len(df)))


def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x


def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[rb] = ra


first_seen = {}
for i, key in enumerate(df["dup_cluster"]):
    key = ("cluster", int(key))
    if key in first_seen:
        union(i, first_seen[key])
    else:
        first_seen[key] = i
for i, key in enumerate(df["group_key"]):
    if isinstance(key, str) and key:
        key = ("explicit", key)
        if key in first_seen:
            union(i, first_seen[key])
        else:
            first_seen[key] = i
df["template_key"] = [template_key(t) if (lab == 1 and shape == "chat") else None
                      for t, lab, shape in zip(df["text"], df["label"], df["shape"])]
for i, key in enumerate(df["template_key"]):
    if key:
        key = ("template", key)
        if key in first_seen:
            union(i, first_seen[key])
        else:
            first_seen[key] = i
df["group"] = [find(i) for i in range(len(df))]

largest = df.groupby("group").agg(n=("text", "size"), source=("source", lambda s: s.value_counts().index[0]),
                                  head=("text", lambda s: s.iloc[0][:70].replace("\n", " "))).sort_values("n", ascending=False).head(10)
print("largest groups:")
print(largest.to_string())
print()

SOURCES = sorted(df["source"].unique().tolist())
SHAPES = sorted(df["shape"].unique().tolist())
BALANCE_KEYS = (["rows", "pos", "neg"] + [f"src::{s}" for s in SOURCES]
                + [f"reason::{r}" for r in REASONS] + [f"shape::{s}" for s in SHAPES])

group_stats = {}
for gid, sub in df.groupby("group"):
    stats = {"rows": len(sub), "pos": int((sub["label"] == 1).sum()), "neg": int((sub["label"] == 0).sum())}
    src_counts = sub["source"].value_counts()
    reason_counts = sub["reason"].value_counts()
    shape_counts = sub["shape"].value_counts()
    for s in SOURCES:
        stats[f"src::{s}"] = int(src_counts.get(s, 0))
    for r in REASONS:
        stats[f"reason::{r}"] = int(reason_counts.get(r, 0))
    for s in SHAPES:
        stats[f"shape::{s}"] = int(shape_counts.get(s, 0))
    group_stats[gid] = stats

totals = {key: sum(stats[key] for stats in group_stats.values()) for key in BALANCE_KEYS}
targets = {split: {key: SPLITS[split] * totals[key] for key in BALANCE_KEYS} for split in SPLITS}
alloc = {split: {key: 0 for key in BALANCE_KEYS} for split in SPLITS}

order = list(group_stats)
np.random.default_rng(SEED).shuffle(order)
order.sort(key=lambda gid: -group_stats[gid]["rows"])


def marginal_cost(split, stats):
    cost = 0.0
    for key in BALANCE_KEYS:
        target = targets[split][key]
        if target <= 0:
            continue
        before = alloc[split][key] - target
        after = before + stats[key]
        cost += (after * after - before * before) / target
    return cost


assign = {}
# Groups with a pre-assigned split (BIPIA documents and templates, section 4) are placed first and
# counted toward the balance; a group must not contain two different forced splits.
forced = df.groupby("group")["forced_split"].agg(lambda s: set(x for x in s if isinstance(x, str)))
for gid, splits in forced.items():
    if splits:
        assert len(splits) == 1, f"group {gid} mixes forced splits {splits}"
        split = next(iter(splits))
        assign[gid] = split
        for key in BALANCE_KEYS:
            alloc[split][key] += group_stats[gid][key]
print(f"groups with a forced split: {len(assign):,}")
for gid in order:
    if gid in assign:
        continue
    stats = group_stats[gid]
    split = min(SPLITS, key=lambda name: marginal_cost(name, stats))
    assign[gid] = split
    for key in BALANCE_KEYS:
        alloc[split][key] += stats[key]

df["split"] = df["group"].map(assign)
print(df.groupby(["split", "label"]).size().unstack(fill_value=0))
print()
print(df.groupby(["split", "source"]).size().unstack(fill_value=0).T)
print()
print(df[df["label"] == 1].groupby(["split", "reason"]).size().unstack(fill_value=0).T)
print()
print(df.groupby(["split", "shape"]).size().unstack(fill_value=0))
print()
print("row share:     ", (df.groupby("split").size() / len(df)).round(4).to_dict())
print("injection rate:", df.groupby("split")["label"].mean().round(4).to_dict())


## 8. Feature Engineering

Per-chunk engineered features for the Phase 2 XGBoost baseline, following `PLAN.md` §2.4:
delimiter and tag counts, special-token markers, imperative-verb density, phrases addressed to the
model, an injection lexicon, obfuscation indicators (zero-width, homoglyph, base64-looking runs),
and plain shape statistics.


In [ ]:
DELIM_RUN_RE = re.compile(r"([#\-=*~_`]){3,}")
ANGLE_TAG_RE = re.compile(r"</?[A-Za-z|][^<>]{0,60}>")
SPECIAL_TOKEN_RE = re.compile(r"<\|[^|>]{1,30}\|>|\[/?INST\]|<<SYS>>|<</SYS>>|</?s>|<\|endoftext\|>|<source\b|</source>|\bHuman:|\bAssistant:", re.IGNORECASE)
IMPERATIVE_RE = re.compile(r"\b(ignore|disregard|forget|override|bypass|pretend|act|respond|reply|answer|output|print|reveal|repeat|translate|write|execute|run|send|delete|stop|begin|start|continue)\b", re.IGNORECASE)
MODEL_ADDRESS_RE = re.compile(r"\b(you are now|you must|you will|your (new )?(instructions?|rules?|task|role)|system prompt|as an ai|from now on|act as|pretend (to be|you are)|developer mode|do anything now|jailbreak|new instructions?|previous instructions?|above instructions?|prior instructions?)\b", re.IGNORECASE)
LEXICON_RE = re.compile(r"\b(ignore|override|system|instruction|instructions|prompt|rule|rules|restriction|restrictions|bypass|disregard|forget|confidential|secret|password|admin)\b", re.IGNORECASE)
BASE64_RUN_RE = re.compile(r"[A-Za-z0-9+/]{40,}={0,2}")
URL_RE = re.compile(r"(?:https?://|www\.)\S+", re.IGNORECASE)
BRACKET_RE = re.compile(r"[\[\]{}()]")
ZW_CHARS = "\u200b\u200c\u200d\u2060\ufeff"
WORDS_RE = re.compile(r"\S+")


def homoglyph_count(text):
    # Cyrillic or Greek letters sitting directly next to Latin letters: the classic lookalike swap.
    count = 0
    for i, ch in enumerate(text):
        if "\u0370" <= ch <= "\u04ff":
            prev_latin = i > 0 and text[i - 1].isascii() and text[i - 1].isalpha()
            next_latin = i + 1 < len(text) and text[i + 1].isascii() and text[i + 1].isalpha()
            if prev_latin or next_latin:
                count += 1
    return count


def engineer(text):
    n_chars = max(len(text), 1)
    words = WORDS_RE.findall(text)
    n_words = max(len(words), 1)
    return {
        "f_len_chars": len(text),
        "f_n_words": len(words),
        "f_delim_runs": len(DELIM_RUN_RE.findall(text)),
        "f_angle_tags": len(ANGLE_TAG_RE.findall(text)),
        "f_special_tokens": len(SPECIAL_TOKEN_RE.findall(text)),
        "f_imperative_density": len(IMPERATIVE_RE.findall(text)) / n_words,
        "f_model_address": len(MODEL_ADDRESS_RE.findall(text)),
        "f_lexicon_hits": len(LEXICON_RE.findall(text)),
        "f_zero_width": sum(c in ZW_CHARS for c in text),
        "f_homoglyph": homoglyph_count(text),
        "f_base64_runs": len(BASE64_RUN_RE.findall(text)),
        "f_url_count": len(URL_RE.findall(text)),
        "f_brackets": len(BRACKET_RE.findall(text)),
        "f_newlines": text.count("\n"),
        "f_upper_ratio": sum(c.isupper() for c in text) / n_chars,
        "f_digit_ratio": sum(c.isdigit() for c in text) / n_chars,
        "f_nonascii_ratio": sum(ord(c) > 127 for c in text) / n_chars,
    }


features = pd.DataFrame([engineer(t) for t in df["text"]])
FEATURE_COLS = features.columns.tolist()
df = pd.concat([df.reset_index(drop=True), features], axis=1)
holdout = pd.concat([holdout.reset_index(drop=True), pd.DataFrame([engineer(t) for t in holdout["text"]])], axis=1)
print(f"{len(FEATURE_COLS)} features")
print(df[FEATURE_COLS].describe().T[["mean", "std", "max"]])


## 9. Adversarial Augmentation (train only)

Deterministic obfuscated copies of a seeded sample of **train** positives, one variant kind per
copy: Cyrillic homoglyph substitution, zero-width injection into trigger words, delimiter wrapping
(fake system block, ChatML, fenced block, `[INST]`), case noise, or whitespace noise. A separate
fixed adversarial evaluation set is built from **test** positives with a different seed; the
originals stay in test and the copies go to `adversarial_test.parquet` only. Features are
recomputed after obfuscation. `aug_kind` records which transform was applied.


In [ ]:
HOMOGLYPHS = {"a": "\u0430", "c": "\u0441", "e": "\u0435", "i": "\u0456", "o": "\u043e", "p": "\u0440", "s": "\u0455", "x": "\u0445"}
ZW = "\u200b"
TRIGGER_RE = re.compile(r"ignore|system|instruction|prompt|override|developer|rules|pretend|disregard|forget|assistant", re.IGNORECASE)
WRAPPERS = [
    "### SYSTEM\n{t}\n### END SYSTEM",
    "---\n{t}\n---",
    "<|im_start|>system\n{t}<|im_end|>",
    "```\n{t}\n```",
    "[INST] {t} [/INST]",
    "<source id=\"1\" name=\"notes.txt\">{t}</source>",
]
AUG_KINDS = ["homoglyph", "zero_width", "delimiter_wrap", "case_noise", "whitespace_noise"]


def obfuscate(text, kind, seed):
    r = np.random.default_rng(seed)
    if kind == "homoglyph":
        out = []
        for ch in text:
            sub = HOMOGLYPHS.get(ch.lower())
            out.append((sub.upper() if ch.isupper() else sub) if sub and r.random() < 0.30 else ch)
        return "".join(out)
    if kind == "zero_width":
        return TRIGGER_RE.sub(lambda m: m.group(0)[:2] + ZW + m.group(0)[2:], text)
    if kind == "delimiter_wrap":
        return WRAPPERS[int(r.integers(len(WRAPPERS)))].format(t=text)
    if kind == "case_noise":
        return "".join((ch.swapcase() if ch.isalpha() and r.random() < 0.15 else ch) for ch in text)
    if kind == "whitespace_noise":
        out = []
        for ch in text:
            if ch == " " and r.random() < 0.20:
                out.append("  " if r.random() < 0.5 else "\n")
            else:
                out.append(ch)
        return "".join(out)
    raise ValueError(kind)


df["augmented"] = False
df["aug_kind"] = ""
holdout["augmented"] = False
holdout["aug_kind"] = ""


def make_adversarial(frame, seed_base):
    adv = frame.copy()
    kinds = [AUG_KINDS[int(k)] for k in np.random.default_rng(seed_base).integers(len(AUG_KINDS), size=len(adv))]
    adv["text"] = [obfuscate(t, k, seed_base + i) for i, (t, k) in enumerate(zip(adv["text"], kinds))]
    adv["aug_kind"] = kinds
    adv["augmented"] = True
    adv[FEATURE_COLS] = pd.DataFrame([engineer(t) for t in adv["text"]], index=adv.index)
    return adv


train_pos = df[(df["split"] == "train") & (df["label"] == 1)]
train_aug = make_adversarial(train_pos.sample(frac=AUG_FRACTION, random_state=SEED), SEED)
test_pos = df[(df["split"] == "test") & (df["label"] == 1)]
adv_eval = make_adversarial(test_pos, SEED + 1000000)
print(f"train augmentations: {len(train_aug):,}  by kind: {train_aug['aug_kind'].value_counts().to_dict()}")
print(f"adversarial eval set: {len(adv_eval):,}  by kind: {adv_eval['aug_kind'].value_counts().to_dict()}")


## 10. Verify and Save

Leakage assertions (no shared group or fingerprint across splits; no held-out fingerprint in any
split; no BIPIA attack template in more than one split), two residual-overlap indicators recorded in
the manifest (template-twin rate of chat-shaped positives, and the share of val/test positives whose
first 12 words appear in train), per-split label, reason and shape distributions, atomic parquet
writes, and a fingerprinted manifest.


In [ ]:
for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    ga = set(df.loc[df["split"] == a, "group"]); gb = set(df.loc[df["split"] == b, "group"])
    assert not (ga & gb), f"group leakage between {a} and {b}"
    fa = set(df.loc[df["split"] == a, "fp"]); fb = set(df.loc[df["split"] == b, "fp"])
    assert not (fa & fb), f"fingerprint leakage between {a} and {b}"
assert not set(holdout["fp"]) & set(df["fp"]), "held-out fingerprint present in the pool"
bipia_tpl = df[df["source"] == "bipia_injected"].groupby("group_key")["split"].nunique()
assert (bipia_tpl <= 1).all(), "a BIPIA attack template spans splits"
bipia_doc = df[df["doc_key"].notna()].groupby("doc_key")["split"].nunique()
assert (bipia_doc <= 1).all(), "a BIPIA base document spans splits (attacked in one, clean in another)"
print(f"BIPIA: {len(bipia_tpl)} attack templates and {len(bipia_doc)} base documents each confined to one split")
print("leakage checks passed")

# Residual-overlap indicator: share of val/test chat-shaped positives whose leading-words key also
# occurs among train positives. Zero by construction for the exact key; reported on a shorter
# 6-word key so skeleton reuse that the 12-word key does not catch stays visible.
short_key = lambda t: template_key(t, words=6)
train_short = set(k for k in df.loc[(df["split"] == "train") & (df["label"] == 1), "text"].map(short_key) if k)
template_twin_rate = {}
for holdout_split in ("val", "test"):
    keys = df.loc[(df["split"] == holdout_split) & (df["label"] == 1) & (df["shape"] == "chat"), "text"].map(short_key)
    keys = keys[keys.notna()]
    rate = float(keys.isin(train_short).mean()) if len(keys) else 0.0
    template_twin_rate[holdout_split] = rate
    print(f"{holdout_split}: 6-word template twin in train = {rate:.1%} ({int(keys.isin(train_short).sum()):,}/{len(keys):,})")

INTERNAL_COLS = ["fp", "dup_cluster", "group", "group_key", "template_key", "split", "holdout", "forced_split", "doc_key"]


def save_atomic(frame, path):
    tmp = Path(str(path) + ".tmp")
    serializable = frame.drop(columns=INTERNAL_COLS, errors="ignore").copy()
    for col in ("text", "reason", "source", "shape", "attack_type", "hf_dataset", "source_id", "aug_kind"):
        serializable[col] = serializable[col].astype("string")
    serializable.to_parquet(tmp, index=False)
    os.replace(tmp, path)


outputs = {
    "train": pd.concat([df[df["split"] == "train"], train_aug], ignore_index=True).sample(frac=1.0, random_state=SEED).reset_index(drop=True),
    "val": df[df["split"] == "val"].reset_index(drop=True),
    "test": df[df["split"] == "test"].reset_index(drop=True),
    "adversarial_test": adv_eval.reset_index(drop=True),
    "lora_holdout": holdout.reset_index(drop=True),
}
counts = {}
for name, frame in outputs.items():
    save_atomic(frame, CLEAN_DIR / f"{name}.parquet")
    counts[name] = {
        "rows": int(len(frame)),
        "injection": int(frame["label"].sum()),
        "safe": int((frame["label"] == 0).sum()),
        "augmented": int(frame["augmented"].sum()),
        "by_source": {k: int(v) for k, v in frame["source"].value_counts().sort_index().items()},
        "by_reason": {k: int(v) for k, v in frame.loc[frame["label"] == 1, "reason"].value_counts().sort_index().items()},
        "by_shape": {k: int(v) for k, v in frame["shape"].value_counts().sort_index().items()},
    }
    print(f"{name}: rows={counts[name]['rows']:,} injection={counts[name]['injection']:,} safe={counts[name]['safe']:,} augmented={counts[name]['augmented']:,}")

manifest = {
    "dataset_version": DATASET_VERSION,
    "config_fingerprint": CONFIG_FINGERPRINT,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "sources": list(HF_DATASETS),
    "excluded_sources": EXCLUDED_SOURCES,
    "reasons": REASONS,
    "reasons_with_training_rows": sorted(outputs["train"].loc[outputs["train"]["label"] == 1, "reason"].unique().tolist()),
    "subcategory_reason_map": SUBCATEGORY_REASON_MAP,
    "v3_changes": [
        "BIPIA document key is the whitespace-normalized base text (v2 hashed raw text and split one document into up to three ids)",
        "injected BIPIA rows capped at three per base document (v2: 48 per attack string, 40:1 injected:clean in train)",
    ],
    "v2_changes": [
        "BIPIA partitioned jointly by base document and attack string; injected rows kept only where both share a split",
        "legitimate emails from puyang2025/seven-phishing-email-datasets added as document-shaped SAFE rows",
        "template key 12 -> 8 leading words; MinHash threshold 0.90 -> 0.80",
    ],
    "labelling_notes": {
        "wildguard_benign_adversarial": "adversarial but unharmful WildGuardMix rows are SAFE, matching the LoRA; kept as a separate slice",
        "nemotron_jailbreaking": "every jailbreaking row is INJECTION regardless of prompt_label, matching the LoRA; prompt_label kept in attack_type",
        "reason_labels": "map-derived from source or subcategory, not human annotated; report, do not gate",
    },
    "holdout_policy": "wildguardtest and Nemotron test-split jailbreaking rows are excluded from every split by exact and near-duplicate match and saved as lora_holdout.parquet",
    "feature_columns": FEATURE_COLS,
    "augmentation_kinds": AUG_KINDS,
    "template_twin_rate_6_words": template_twin_rate,
    "largest_group_sizes": [int(n) for n in largest["n"].tolist()],
    "counts": counts,
}
tmp = CLEAN_DIR / "manifest.json.tmp"
tmp.write_text(json.dumps(manifest, indent=2))
os.replace(tmp, CLEAN_DIR / "manifest.json")
print("manifest written:", CLEAN_DIR / "manifest.json")


## 11. Report

Writes `reports/<UTC stamp>_phase_1_datagen_<version>_<fingerprint>/report.md`: sources, per-split counts by label,
source, reason and shape, the leakage indicators, and a reason-distribution chart beside it. `reports/` is tracked in git; `data/` is not, so this is the part of the build
that travels with the repo.


In [ ]:
# ---- Report: tracked Markdown summary of this run, written to reports/ with its plots beside it ----
import shutil

REPORTS_DIR = PROJECT_DIR / "reports"
RUN_STAMP = datetime.now(timezone.utc).strftime("%Y-%m-%d_%H%M%SZ")


def report_dir(run_name):
    # One folder per run: reports/<UTC stamp>_<phase>_<version>_<fingerprint>/ holding report.md and its plots.
    path = REPORTS_DIR / f"{RUN_STAMP}_{run_name}"
    path.mkdir(parents=True, exist_ok=True)
    return path


def md_table(frame, floatfmt="{:.4f}", index=True):
    frame = pd.DataFrame(frame)
    cols = [str(c) for c in frame.columns]
    header = ([""] if index else []) + cols
    lines = ["| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    for idx, row in frame.iterrows():
        cells = []
        for v in row.tolist():
            if v is None or (isinstance(v, float) and np.isnan(v)):
                cells.append("")
            elif isinstance(v, (float, np.floating)):
                cells.append(floatfmt.format(float(v)))
            elif isinstance(v, (bool, np.bool_)):
                cells.append("yes" if v else "no")
            else:
                cells.append(str(v).replace("|", "\\|").replace("\n", " "))
        lines.append("| " + " | ".join(([str(idx)] if index else []) + cells) + " |")
    return "\n".join(lines)


def write_report(name, title, sections):
    # sections: list of (heading, markdown body). Atomic write, LF endings.
    parts = [f"# {title}", f"_Generated {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M UTC')}_"]
    for heading, body in sections:
        parts.append(f"## {heading}\n\n{body}" if heading else body)
    path = report_dir(name) / "report.md"
    tmp = Path(str(path) + ".tmp")
    tmp.write_text("\n\n".join(parts) + "\n", encoding="utf-8", newline="\n")
    os.replace(tmp, path)
    print("report written:", path)
    return path


def report_asset(run_name, filename):
    # Plots live beside report.md and are referenced by bare file name.
    return report_dir(run_name), filename

run_name = f"phase_1_datagen_{DATASET_VERSION}_{CONFIG_FINGERPRINT[:16]}"
split_names = ["train", "val", "test", "adversarial_test", "lora_holdout"]
by_label = pd.DataFrame({s: {k: counts[s][k] for k in ("rows", "injection", "safe", "augmented")} for s in split_names}).T
by_source = pd.DataFrame({s: counts[s]["by_source"] for s in split_names}).fillna(0).astype(int)
by_reason = pd.DataFrame({s: counts[s]["by_reason"] for s in split_names}).reindex(REASONS).fillna(0).astype(int)
by_shape = pd.DataFrame({s: counts[s]["by_shape"] for s in split_names}).fillna(0).astype(int)

chart_line = "_matplotlib not installed in this kernel; chart skipped._"
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    asset_dir, rel = report_asset(run_name, "reason_distribution.png")
    axis = by_reason[["train", "val", "test"]].plot.barh(figsize=(9, 5), title="Injection rows per reason, by split")
    axis.set_xlabel("rows")
    plt.tight_layout()
    plt.savefig(asset_dir / "reason_distribution.png", dpi=140)
    plt.close()
    chart_line = f"![reason distribution]({rel})"
except Exception as exc:
    chart_line = f"_chart skipped: {type(exc).__name__}: {exc}_"
    print("chart skipped:", exc)

sources_md = "\n".join(f"- `{s}`" for s in HF_DATASETS)
excluded_md = "\n".join(f"- `{k}`: {v}" for k, v in EXCLUDED_SOURCES.items())
write_report(run_name, f"Phase 1 Report: Dataset {DATASET_VERSION} ({CONFIG_FINGERPRINT[:16]})", [
    ("Build", f"- Dataset version: `{DATASET_VERSION}`\n- Config fingerprint: `{CONFIG_FINGERPRINT}`\n- Created: {manifest['created_utc']}\n- Output: `{CLEAN_DIR.relative_to(PROJECT_DIR)}`\n- Reasons with training rows: {', '.join(manifest['reasons_with_training_rows'])}"),
    ("Sources", sources_md + "\n\nExcluded:\n\n" + excluded_md),
    ("Rows per split", md_table(by_label, floatfmt="{:.0f}")),
    ("Rows per source and split", md_table(by_source, floatfmt="{:.0f}")),
    ("Injection rows per reason and split", md_table(by_reason, floatfmt="{:.0f}") + "\n\n" + chart_line),
    ("Rows per shape and split", md_table(by_shape, floatfmt="{:.0f}")),
    ("Leakage indicators", "- Cross-split group and fingerprint overlap: none (asserted)\n- Held-out rows removed from the pool: exact and near-duplicate match (asserted)\n"
                           + "\n".join(f"- 6-word template twin rate, {k}: {v:.1%}" for k, v in template_twin_rate.items())
                           + "\n- Ten largest split groups (rows): " + ", ".join(str(n) for n in manifest["largest_group_sizes"])),
    ("Labelling notes", "\n".join(f"- **{k}**: {v}" for k, v in manifest["labelling_notes"].items())),
])
